# K-Means na prática: segmentação de clientes (RFM)
Envie o arquivo `compras.csv` quando a célula 2 pedir.

In [ ]:
# 1. Importar bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from google.colab import files

In [ ]:
# 2. Enviar o arquivo compras.csv
files.upload()
df = pd.read_csv("compras.csv", parse_dates=["InvoiceDate"])
df.head()

In [ ]:
# 3. Montar a tabela RFM (uma linha por cliente)
df["Total"] = df["Quantity"] * df["UnitPrice"]
ref = df["InvoiceDate"].max()

rfm = df.groupby("CustomerID").agg(
    Recencia=("InvoiceDate", lambda d: (ref - d.max()).days),
    Frequencia=("InvoiceNo", "nunique"),
    Monetario=("Total", "sum"))
rfm.head()

In [ ]:
# 4. Padronizar os dados
X = StandardScaler().fit_transform(rfm)

In [ ]:
# 5. Método do cotovelo: qual K escolher?
wcss = []
for k in range(1, 9):
    wcss.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_)

plt.plot(range(1, 9), wcss, marker="o")
plt.xlabel("Número de clusters (K)")
plt.ylabel("WCSS (inércia)")
plt.title("Método do cotovelo")
plt.show()

In [ ]:
# 6. Aplicar o K-Means com K = 3
modelo = KMeans(n_clusters=3, init="k-means++", random_state=42, n_init=10)
rfm["Cluster"] = modelo.fit_predict(X)

# 7. Interpretar: média de R, F e M de cada grupo
rfm.groupby("Cluster").mean().round(1)

In [ ]:
# 8. Visualizar os grupos
plt.scatter(rfm["Frequencia"], rfm["Monetario"], c=rfm["Cluster"], cmap="viridis")
plt.xlabel("Frequência (nº de compras)")
plt.ylabel("Monetário (R$)")
plt.title("Clientes agrupados pelo K-Means")
plt.show()